# Class 9 - NumPy for Analytics
### Data Analytics in Python | Week 3, Sunday

The previous two classes gave you the tools. This one shows you how to
**use them to answer analytical questions**.

Analytics in NumPy means: filter, rank, count, simulate, and measure
relationships - all without writing explicit loops, because NumPy gives
you better ways for every one of those tasks.

**Today:**
- Boolean indexing and fancy (integer array) indexing
- `np.where()` - vectorized conditional assignment
- `np.sort()` and `np.argsort()` - sorting and ranking
- `np.unique()` and `np.bincount()` - frequency analysis
- `np.random` - reproducible simulation
- `np.percentile()`, `np.corrcoef()`, `np.histogram()`
- Practical: Exam score analytics engine (30 students × 5 subjects)

## 1. Boolean indexing - the analytics workhorse

You saw a preview in Class 7. Today we go deeper. Boolean indexing is how NumPy replaces most filtering loops.

In [2]:
import numpy as np

# Patient glucose readings
glucose = np.array([5.1, 9.4, 4.8, 11.3, 6.2, 8.7, 5.6, 12.1, 4.9, 7.3])

# Step 1: a condition creates a boolean array
high_mask = glucose > 7.0
print(f"glucose:   {glucose}")
print(f"mask >7.0: {high_mask}")

# Step 2: use the mask to filter
high_glucose = glucose[high_mask]
print(f"High readings: {high_glucose}")

# Compound conditions - use & (and) and | (or), NOT following 'and'/'or'
borderline = glucose[(glucose >= 7.0) & (glucose < 11.0)]
critical   = glucose[glucose >= 11.0]
print(f"Borderline [7, 11): {borderline}")
print(f"Critical   ≥11.0:   {critical}")

# Count and percentage
n_high = high_mask.sum()
print(f"\nHigh glucose: {n_high}/{len(glucose)} = {n_high/len(glucose)*100:.0f}%")

glucose:   [ 5.1  9.4  4.8 11.3  6.2  8.7  5.6 12.1  4.9  7.3]
mask >7.0: [False  True False  True False  True False  True False  True]
High readings: [ 9.4 11.3  8.7 12.1  7.3]
Borderline [7, 11): [9.4 8.7 7.3]
Critical   ≥11.0:   [11.3 12.1]

High glucose: 5/10 = 50%


In [3]:
# Boolean indexing for assignment - modify selected elements
readings = np.array([37.2, 38.9, -99.0, 36.5, 39.4, -99.0, 37.8])
print(f"Before: {readings}")

# Replace sentinel values (-99) with NaN
readings[readings == -99.0] = np.nan
print(f"After:  {readings}")

# Now compute stats ignoring NaN
valid = readings[~np.isnan(readings)]      #Bitwise NOT (~) - tilde inverts the bits of an integer, same as not operator
print(f"Valid:  {valid}")
print(f"Mean of valid: {valid.mean():.2f}")

# 2D boolean indexing - same logic works on matrices
scores = np.array([[88, 55, 94],
                   [43, 77, 91],
                   [60, 82, 38]])
scores[scores < 50] = 50   # floor all failing scores at 50
print(f"\nAfter flooring below 50:\n{scores}")

Before: [ 37.2  38.9 -99.   36.5  39.4 -99.   37.8]
After:  [37.2 38.9  nan 36.5 39.4  nan 37.8]
Valid:  [37.2 38.9 36.5 39.4 37.8]
Mean of valid: 37.96

After flooring below 50:
[[88 55 94]
 [50 77 91]
 [60 82 50]]


## 2. Fancy indexing - select specific positions

In [4]:
data = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90])

# Fancy indexing: pass an array of indices
indices = np.array([0, 2, 5, 7])
selected = data[indices]
print(f"data:       {data}")
print(f"indices:    {indices}")
print(f"selected:   {selected}")

# Unlike slicing, fancy indexing returns a COPY
selected[0] = 999
print(f"\nAfter modifying selected[0]:")
print(f"  selected: {selected}")
print(f"  data:     {data}")   # unchanged

# 2D fancy indexing - select specific rows
matrix = np.arange(20).reshape(4, 5)
print(f"\nMatrix:\n{matrix}")

row_indices = np.array([0, 3, 1])   # pick rows 0, 3, 1 in that order
print(f"\nRows [0,3,1]:\n{matrix[row_indices, :]}")

data:       [10 20 30 40 50 60 70 80 90]
indices:    [0 2 5 7]
selected:   [10 30 60 80]

After modifying selected[0]:
  selected: [999  30  60  80]
  data:     [10 20 30 40 50 60 70 80 90]

Matrix:
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

Rows [0,3,1]:
[[ 0  1  2  3  4]
 [15 16 17 18 19]
 [ 5  6  7  8  9]]


## 3. np.where() - vectorized if/else on arrays

In [5]:
scores = np.array([88, 55, 94, 43, 77, 91, 60, 82, 38, 96])

# np.where(condition, value_if_true, value_if_false)
labels = np.where(scores >= 60, "pass", "fail")
print(f"scores: {scores}")
print(f"labels: {labels}")

# Nested np.where for multi-category classification
grades = np.where(scores >= 90, "A",
         np.where(scores >= 80, "B",
         np.where(scores >= 70, "C",
         np.where(scores >= 60, "D", "F"))))
print(f"grades: {grades}")

scores: [88 55 94 43 77 91 60 82 38 96]
labels: ['pass' 'fail' 'pass' 'fail' 'pass' 'pass' 'pass' 'pass' 'fail' 'pass']
grades: ['B' 'F' 'A' 'F' 'C' 'A' 'D' 'B' 'F' 'A']


In [6]:
np.random.seed(42)

# Clinical: flag patients based on multiple thresholds
temp     = np.random.uniform(36.0, 40.5, 20)
bp_sys   = np.random.uniform(110, 170, 20)
glucose  = np.random.uniform(4.5, 13.0, 20)

# Risk score: assign points based on severity
risk = (
    np.where(temp >= 39.5, 3, np.where(temp >= 38.0, 2, np.where(temp >= 37.5, 1, 0))) +
    np.where(bp_sys >= 160, 3, np.where(bp_sys >= 140, 2, np.where(bp_sys >= 130, 1, 0))) +
    np.where(glucose >= 11.0, 2, np.where(glucose >= 7.0, 1, 0))
)

levels = np.where(risk >= 6, "CRITICAL",
         np.where(risk >= 4, "HIGH",
         np.where(risk >= 2, "MEDIUM", "LOW")))

for level in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    count = (levels == level).sum()
    print(f"  {level:<10} {count:3d} patients")

  CRITICAL     1 patients
  HIGH         9 patients
  MEDIUM       7 patients
  LOW          3 patients


In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# # temp     = np.random.uniform(36.0, 40.5, 100000)
# guassian = np.random.randn(36, 40, 100000)
 
# plt.figure(figsize=(6, 4))
# sns.kdeplot(guassian, fill=True, bw_adjust=0.5)
# plt.title("Uniform Distribution - Density Plot")
# plt.xlabel("Value")
# plt.ylabel("Density")
# plt.show()

## 4. Sorting and ranking with argsort()

In [3]:
scores = np.array([88, 55, 94, 43, 77, 91, 60, 82, 38, 96])
names  = ["Ahmad","Sara","Bilal","Zara","Hassan","Nida","Usman","Ayesha","Faisal","Mariam"]
names  = np.array(names)

# np.sort - returns sorted VALUES (does not modify original)
sorted_asc  = np.sort(scores)
sorted_desc = np.sort(scores)[::-1]
print(f"Sorted asc:  {sorted_asc}")
print(f"Sorted desc: {sorted_desc}")

# np.argsort - returns INDICES that would sort the array
# (the indices, not the values - this trips people up)
rank_indices = np.argsort(scores)          # ascending
rank_desc    = np.argsort(scores)[::-1]   # descending

print(f"\nargsort (ascending):  {rank_indices}")
print(f"argsort (descending): {rank_desc}")

# Use indices to reorder NAMES by score
print(f"\nStudents ranked by score (best to worst):")
for rank, idx in enumerate(rank_desc, 1):
    print(f"  {rank:2d}. {names[idx]:<12} {scores[idx]}")

Sorted asc:  [38 43 55 60 77 82 88 91 94 96]
Sorted desc: [96 94 91 88 82 77 60 55 43 38]

argsort (ascending):  [8 3 1 6 4 7 0 5 2 9]
argsort (descending): [9 2 5 0 7 4 6 1 3 8]

Students ranked by score (best to worst):
   1. Mariam       96
   2. Bilal        94
   3. Nida         91
   4. Ahmad        88
   5. Ayesha       82
   6. Hassan       77
   7. Usman        60
   8. Sara         55
   9. Zara         43
  10. Faisal       38


### `argsort on 2D`: rank students within each subject

In [ ]:
# # argsort on 2D: rank students within each subject
# np.random.seed(7)
# scores_2d = np.random.randint(40, 101, (8, 4))
# subjects  = ["Math", "Python", "Stats", "DS"]
# students  = [f"S{i+1:02d}" for i in range(8)]

# print("Score matrix (students x subjects):")
# print(f"{'':>5} " + " ".join(f"{s:>7}" for s in subjects))
# for i, row in enumerate(scores_2d):
#     print(f"{students[i]:>5} " + " ".join(f"{v:>7}" for v in row))

# # Rank students in Python (col 1) - best to worst
# python_ranks = np.argsort(scores_2d[:, 1])[::-1]
# print(f"Python subject ranking:")
# for rank, idx in enumerate(python_ranks, 1):
#     print(f"  Rank {rank}: {students[idx]}  score={scores_2d[idx, 1]}")

Score matrix (students x subjects):
         Math  Python   Stats      DS
  S01      87      44      65      94
  S02      43      59      63      79
  S03      68      97      54      63
  S04      48      65      86      82
  S05      66      48      79      78
  S06      44      88      47      84
  S07      40      51      95      98
  S08      46      59     100      84
Python subject ranking:
  Rank 1: S03  score=97
  Rank 2: S06  score=88
  Rank 3: S04  score=65
  Rank 4: S08  score=59
  Rank 5: S02  score=59
  Rank 6: S07  score=51
  Rank 7: S05  score=48
  Rank 8: S01  score=44


## 5. np.unique() and np.bincount() - frequency analysis

In [9]:
# np.unique - find distinct values
reads = np.array([1, 3, 3, 2, 1, 4, 3, 2, 1, 5, 5, 1])

unique_vals = np.unique(reads)
print(f"reads:      {reads}")
print(f"unique:     {unique_vals}")

# With return_counts=True - frequency for each unique value
vals, counts = np.unique(reads, return_counts=True) #returns unique values and their counts also
print(f"Value --> Count:")
for v, c in zip(vals, counts):
    print(f"  {v}: {c}  {'█' * c}")

reads:      [1 3 3 2 1 4 3 2 1 5 5 1]
unique:     [1 2 3 4 5]
Value --> Count:
  1: 4  ████
  2: 2  ██
  3: 3  ███
  4: 1  █
  5: 2  ██


In [8]:
# np.bincount - fastest frequency for non-negative integers
# Produces an array where index i = count of value i
grades_numeric = np.array([2, 0, 1, 2, 3, 1, 0, 2, 4, 1, 2, 0])
# 0=F, 1=D, 2=C, 3=B, 4=A

counts = np.bincount(grades_numeric)
labels = ["F", "D", "C", "B", "A"]

print("Grade frequency (bincount):")
for i, (label, count) in enumerate(zip(labels, counts)):
    bar = "█" * count
    print(f"  {label} ({i}): {bar} ({count})")

# np.unique with return_counts=True works on ANY type (including strings)
gene_hits = np.array(["BRCA1","TP53","KRAS","BRCA1","TP53","BRCA1","EGFR"])
genes, hit_counts = np.unique(gene_hits, return_counts=True)
ranked = np.argsort(hit_counts)[::-1]

print("Gene hit frequency:")
for i in ranked:
    print(f"  {genes[i]:<8} {hit_counts[i]}")

Grade frequency (bincount):
  F (0): ███ (3)
  D (1): ███ (3)
  C (2): ████ (4)
  B (3): █ (1)
  A (4): █ (1)
Gene hit frequency:
  BRCA1    3
  TP53     2
  KRAS     1
  EGFR     1


## 6. np.random - reproducible simulation

In [ ]:
# ALWAYS SET A SEED before any simulation
np.random.seed(42)

# Integer random values
dice = np.random.randint(1, 7, size=10)   # 10 dice rolls
print(f"10 dice rolls: {dice}")

# Uniform random in [0, 1)
uniform = np.random.random(5)
print(f"Uniform [0,1): {uniform.round(4)}")

# Normal distribution
heights = np.random.normal(loc=170, scale=10, size=8)   # mean=170, std=10
print(f"Heights (cm):  {heights.round(1)}")

# Shuffle (in-place)
arr = np.arange(1, 11)
np.random.shuffle(arr)
print(f"Shuffled 1-10: {arr}")

# Sample without replacement
pool = np.array(["P001","P002","P003","P004","P005","P006","P007","P008"])
selected = np.random.choice(pool, size=4, replace=False)        #replace=False: Ensures sampling without replacement. Once an item from pool is picked, it cannot be picked again. All 4 selected items will be strictly unique.
print(f"Selected patients: {selected}")

10 dice rolls: [4 5 3 5 5 2 3 3 3 5]
Uniform [0,1): [0.6011 0.7081 0.0206 0.9699 0.8324]
Heights (cm):  [164.2 164.7 164.3 160.8 143.9 179.5 178.2 154.8]
Shuffled 1-10: [ 2  6  5  9  1  8  7  4  3 10]
Selected patients: ['P007' 'P005' 'P001' 'P007']


In [18]:
# Reproducibility: same seed = same numbers
for seed in [42, 42, 99]:
    np.random.seed(seed)
    sample = np.random.randint(0, 100, 5)
    print(f"seed={seed}: {sample}")

# Monte Carlo simulation: estimate pi
np.random.seed(2024)
N = 1_000_000
x = np.random.uniform(-1, 1, N)
y = np.random.uniform(-1, 1, N)
inside_circle = (x**2 + y**2) <= 1.0
pi_estimate = 4 * inside_circle.sum() / N

print(f"\nMonte Carlo pi estimate ({N:,} samples):")
print(f"  Estimated: {pi_estimate:.6f}")
print(f"  True pi:   {np.pi:.6f}")
print(f"  Error:     {abs(pi_estimate - np.pi):.6f}")

seed=42: [51 92 14 71 60]
seed=42: [51 92 14 71 60]
seed=99: [ 1 35 57 40 73]

Monte Carlo pi estimate (1,000,000 samples):
  Estimated: 3.145552
  True pi:   3.141593
  Error:     0.003959


### 6b. Slicing in High-Dimensional Spaces (3D+)
In advanced analytics and deep learning (e.g., Computer Vision or Time-Series forecasting), arrays span 3 or more dimensions (e.g., `[batches, rows, columns]`). Slicing follows the exact same comma-separated logic, extended to every dimension.

In [42]:
# Time-series data: 2 patient batches, tracking 3 vitals, over 4 hours
# Shape: (2, 3, 4) -> [batches, features, time_steps]
timeseries = np.arange(24).reshape(2, 3, 4)
print(f"3D Array Shape: {timeseries.shape}, Data:\n {timeseries}")
print('-'*50)

# Extract: All batches (:), only the first feature (0), all time steps (:)
first_feature = timeseries[:, 0, :]
print(f"\nFirst feature across all batches:\n{first_feature}")

# Extract: Batch 1 (1), first two features (:2), last two time steps (2:)
subset = timeseries[1, :2, 2:]
print(f"\nSpecific window subset:\n{subset}")

3D Array Shape: (2, 3, 4), Data:
 [[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]
--------------------------------------------------

First feature across all batches:
[[ 0  1  2  3]
 [12 13 14 15]]

Specific window subset:
[[14 15]
 [18 19]]


## 7. Statistical functions for analytics

In [19]:
data = np.array([88, 55, 94, 43, 77, 91, 60, 82, 38, 96, 72, 68, 85, 58, 79])

# Percentiles
p25, p50, p75 = np.percentile(data, [25, 50, 75])
iqr = p75 - p25
print(f"Data: {data}")
print(f"Q1 (25th):   {p25}")
print(f"Median (50th):{p50}")
print(f"Q3 (75th):   {p75}")
print(f"IQR:         {iqr}")

# Outlier fences using IQR
lo_fence = p25 - 1.5 * iqr
hi_fence = p75 + 1.5 * iqr
outliers = data[(data < lo_fence) | (data > hi_fence)]
print(f"Outlier fences: [{lo_fence:.1f}, {hi_fence:.1f}]")
print(f"Outliers: {outliers}")

Data: [88 55 94 43 77 91 60 82 38 96 72 68 85 58 79]
Q1 (25th):   59.0
Median (50th):77.0
Q3 (75th):   86.5
IQR:         27.5
Outlier fences: [17.8, 127.8]
Outliers: []


### 7b. Dealing with Missing Data (NaN-Safe Operations)
Real-world data is messy and often contains missing values represented by `np.nan`. Standard statistical functions break down completely when encountering a single `nan`. NumPy provides specialized "NaN-safe" alternatives to bypass this behavior.

In [43]:
#data with missing readings
vitals = np.array([120.0, 115.0, np.nan, 128.0, np.nan])
print(f"Raw vitals array: {vitals}")

# Standard operations fail gracefully but uselessly
print(f"Standard mean: {np.mean(vitals)}")

# NaN-safe operations ignore the missing blocks completely
safe_mean = np.nanmean(vitals)
safe_std  = np.nanstd(vitals)
print(f"NaN-safe mean: {safe_mean:.2f}")
print(f"NaN-safe std:  {safe_std:.2f}")

Raw vitals array: [120. 115.  nan 128.  nan]
Standard mean: nan
NaN-safe mean: 121.00
NaN-safe std:  5.35


In [20]:
# np.corrcoef - Pearson correlation matrix
study_hours = np.array([2, 3, 5, 4, 6, 7, 8, 5, 3, 9])
exam_scores = np.array([55, 60, 75, 70, 80, 85, 92, 72, 58, 95])
sleep_hours = np.array([8, 7, 6, 7, 5, 5, 4, 6, 7, 3])

# Stack into matrix (each row = one variable)
matrix = np.array([study_hours, exam_scores, sleep_hours])
corr   = np.corrcoef(matrix)

labels = ["Study hrs", "Exam score", "Sleep hrs"]
print("Pearson correlation matrix:")
print(f"{'':>12}" + "".join(f"{l:>12}" for l in labels))
for i, row in enumerate(corr):
    print(f"{labels[i]:>12}" + "".join(f"{v:>12.4f}" for v in row))

# Direct 2-variable correlation
r = np.corrcoef(study_hours, exam_scores)[0, 1]
print(f"Study hours vs Exam scores: r = {r:.4f}")

Pearson correlation matrix:
               Study hrs  Exam score   Sleep hrs
   Study hrs      1.0000      0.9914     -0.9855
  Exam score      0.9914      1.0000     -0.9634
   Sleep hrs     -0.9855     -0.9634      1.0000
Study hours vs Exam scores: r = 0.9914


In [21]:
# np.histogram - count values falling in bins
np.random.seed(42)
bmi_values = np.random.normal(25.5, 4.5, 500).clip(15, 45)

counts, bin_edges = np.histogram(bmi_values, bins=6)

print("BMI Distribution (500 patients):")
for i, (lo, hi, c) in enumerate(zip(bin_edges[:-1], bin_edges[1:], counts)):
    bar = "█" * (c // 5)
    print(f"  [{lo:5.1f}-{hi:5.1f}): {bar} ({c})")

# Percentile-based bin edges
pct_bins = np.percentile(bmi_values, [0, 25, 50, 75, 100])
counts_pct, _ = np.histogram(bmi_values, bins=pct_bins)
print(f"Equal-frequency bins (quartile-based):")
for i, (lo, hi, c) in enumerate(zip(pct_bins[:-1], pct_bins[1:], counts_pct)):
    label = ["Underweight/Normal","Normal/Overweight","Overweight","Obese"][i]
    print(f"  {label:<22} {c} patients")

BMI Distribution (500 patients):
  [ 15.0- 19.6): █████████ (45)
  [ 19.6- 24.3): █████████████████████████████ (148)
  [ 24.3- 28.9): ████████████████████████████████████████ (201)
  [ 28.9- 33.6): █████████████████ (86)
  [ 33.6- 38.2): ███ (18)
  [ 38.2- 42.8):  (2)
Equal-frequency bins (quartile-based):
  Underweight/Normal     125 patients
  Normal/Overweight      125 patients
  Overweight             125 patients
  Obese                  125 patients


## Practical

### Practical 1 - argsort returns INDICES, not values
**Context:** A student tries to display the top 3 scoring genes but instead displays the top 3 indices. Classic confusion between argsort output and what to do with it.

In [22]:
expression = np.array([1.23, 3.87, 0.91, 2.45, 4.12, 1.76, 3.34, 0.62])
genes      = np.array(["BRCA1","TP53","KRAS","EGFR","MYC","RB1","CDK4","PTEN"])

# What argsort actually returns
rank_idx = np.argsort(expression)[::-1]   # indices sorted by expression (desc)

print("expression values:", expression)
print("argsort desc:     ", rank_idx)      # INDICES, not values!
print()
print("What most students think they got:")
print(f"  'top 3': {rank_idx[:3]}")        # [4, 1, 6] - these are indices!
print()
print("What they actually wanted:")
print(f"  Top 3 gene NAMES:   {genes[rank_idx[:3]]}")
print(f"  Top 3 EXPRESSION:   {expression[rank_idx[:3]]}")
print()
print("Rule: argsort gives you INDICES. Use those indices to fetch VALUES.")

expression values: [1.23 3.87 0.91 2.45 4.12 1.76 3.34 0.62]
argsort desc:      [4 1 6 3 5 0 2 7]

What most students think they got:
  'top 3': [4 1 6]

What they actually wanted:
  Top 3 gene NAMES:   ['MYC' 'TP53' 'CDK4']
  Top 3 EXPRESSION:   [4.12 3.87 3.34]

Rule: argsort gives you INDICES. Use those indices to fetch VALUES.


### Practical 2 - Boolean mask vs fancy indexing: which modifies in-place?
**Context:** An analyst wants to zero out all failing scores in a dataset. They try two approaches - one works, one silently does nothing.

In [26]:
scores = np.array([88, 45, 92, 38, 77, 55, 60])
print(f"Original: {scores}")

# --- Fancy indexing on a boolean result creates a COPY ---
failing_indices = np.where(scores < 60)[0]   # array of indices
copy_of_failing = scores[failing_indices]     # this is a COPY

copy_of_failing[:] = 0    # modify the copy
print(f"After modifying copy_of_failing: {scores}")   # UNCHANGED!

# --- Direct boolean mask assignment modifies IN-PLACE ---
scores2 = np.array([88, 45, 92, 38, 77, 55, 60])
scores2[scores2 < 60] = 0    # boolean mask on LEFT SIDE = in-place
print(f"After boolean mask assignment:   {scores2}")   # CHANGED

# Key rule:
# array[mask]      --> creates a COPY (fancy indexing on right-hand side)
# array[mask] = v  --> modifies IN-PLACE (mask on left-hand side of assignment)


Original: [88 45 92 38 77 55 60]
After modifying copy_of_failing: [88 45 92 38 77 55 60]
After boolean mask assignment:   [88  0 92  0 77  0 60]


### Practical 3 - np.random.seed: why reproducibility is non-negotiable in science
**Context:** Two researchers run the 'same' simulation. They get different results. Their paper can't be reproduced. Why?

In [27]:
# Without seed - different every run
results_no_seed = [np.random.randint(0, 100, 5) for _ in range(3)]
print("Without seed (different every run):")
for r in results_no_seed:
    print(f"  {r}")

# With seed - reproducible
results_seeded = []
for _ in range(3):
    np.random.seed(2024)
    results_seeded.append(np.random.randint(0, 100, 5))
print("With seed=2024 (same every run):")
for r in results_seeded:
    print(f"  {r}")

# Real consequence: drug trial simulation
np.random.seed(42)
group_A = np.random.normal(140, 15, 50)   # control BP
group_B = np.random.normal(135, 15, 50)   # treatment BP

from scipy import stats as st
t_stat, p_value = st.ttest_ind(group_A, group_B)
print(f"Drug trial simulation:")
print(f"  Control mean:   {group_A.mean():.2f} mmHg")
print(f"  Treatment mean: {group_B.mean():.2f} mmHg")
print(f"  p-value:        {p_value:.4f}")
print(f"  Conclusion:     {'Significant' if p_value < 0.05 else 'Not significant'}")
print("Same seed = same patients = same conclusion every time = reproducible science")

Without seed (different every run):
  [25 63 97 58 55]
  [58 69 32 52 21]
  [20 69 69  3 93]
With seed=2024 (same every run):
  [ 8 96  0 27 36]
  [ 8 96  0 27 36]
  [ 8 96  0 27 36]
Drug trial simulation:
  Control mean:   136.62 mmHg
  Treatment mean: 135.27 mmHg
  p-value:        0.6196
  Conclusion:     Not significant
Same seed = same patients = same conclusion every time = reproducible science


### Practical 4 - np.percentile for outlier detection vs the Z-score method
**Context:** A salary dataset has a few very high earners. Which outlier method catches them? Which is more robust?

In [28]:
# Salary data: 20 employees, 3 executives with very high salaries
np.random.seed(42)
salaries = np.concatenate([
    np.random.normal(45000, 8000, 17),    # regular employees
    np.array([150000, 175000, 220000])    # executives
])

print(f"Salary data ({len(salaries)} employees):")
print(f"  Mean:   {salaries.mean():,.0f}")
print(f"  Median: {np.median(salaries):,.0f}")
print(f"  Std:    {salaries.std():,.0f}")

# Method 1: IQR (robust - based on percentiles)
q1, q3 = np.percentile(salaries, [25, 75])
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
iqr_outliers = salaries[(salaries < lo) | (salaries > hi)]

# Method 2: Z-score (sensitive to the outliers themselves)
z = (salaries - salaries.mean()) / salaries.std()
z_outliers = salaries[np.abs(z) > 2]

print(f"IQR method  [fence: {lo:,.0f} – {hi:,.0f}]:")
print(f"  Outliers ({len(iqr_outliers)}): {iqr_outliers.astype(int)}")

print(f"Z-score method (|z| > 2):")
print(f"  Outliers ({len(z_outliers)}): {z_outliers.astype(int)}")

print("Lesson: IQR is more robust - it uses medians, not means")
print("Z-score is pulled by the very outliers it's trying to detect")

Salary data (20 employees):
  Mean:   64,932
  Median: 45,415
  Std:    50,782
IQR method  [fence: 24,191 – 69,727]:
  Outliers (3): [150000 175000 220000]
Z-score method (|z| > 2):
  Outliers (2): [175000 220000]
Lesson: IQR is more robust - it uses medians, not means
Z-score is pulled by the very outliers it's trying to detect


## 9. Practical - Exam Score Analytics Engine

In [3]:
# 30 students x 5 subjects
np.random.seed(99)
subjects  = ["Mathematics", "Python", "Statistics", "Data Science", "Communication"]
scores    = np.random.randint(35, 101, (30, 5)).astype(float)
student_names = np.array([f"Student_{i+1:02d}" for i in range(30)])

print(f"Score matrix: {scores.shape} - {scores.shape[0]} students x {scores.shape[1]} subjects")
print(f"Score range:  [{scores.min():.0f}, {scores.max():.0f}]")

Score matrix: (30, 5) - 30 students x 5 subjects
Score range:  [35, 100]


In [4]:
# Per-subject statistics (axis=0 = collapse rows = per column)
print("=== Subject Statistics ===")
print(f"{'Subject':<16} {'Mean':>6} {'Std':>6} {'Min':>5} {'Max':>5} {'<60':>5}")
print("-" * 45)
for i, subj in enumerate(subjects):
    col   = scores[:, i]
    fails = (col < 60).sum()
    print(f"{subj:<16} {col.mean():>6.1f} {col.std():>6.1f} "
          f"{col.min():>5.0f} {col.max():>5.0f} {fails:>5}")

=== Subject Statistics ===
Subject            Mean    Std   Min   Max   <60
---------------------------------------------
Mathematics        60.8   16.6    35   100    14
Python             69.2   16.2    36    97    10
Statistics         72.0   21.6    35    99    11
Data Science       67.8   18.5    38    99    13
Communication      67.7   19.9    35   100    10


In [5]:
# Per-student totals and ranking (axis=1 = collapse cols = per row)
totals = scores.sum(axis=1)
avgs   = scores.mean(axis=1)

# Rank students: argsort gives ascending, reverse for descending
rank_idx = np.argsort(avgs)[::-1]

print("=== Top 5 and Bottom 5 Students ===")
print("Top 5:")
for pos, idx in enumerate(rank_idx[:5], 1):
    print(f"  {pos}. {student_names[idx]}  avg={avgs[idx]:.1f}  total={totals[idx]:.0f}")

print("Bottom 5:")
for pos, idx in enumerate(rank_idx[-5:][::-1], 1):
    print(f"  {pos}. {student_names[idx]}  avg={avgs[idx]:.1f}  total={totals[idx]:.0f}")

=== Top 5 and Bottom 5 Students ===
Top 5:
  1. Student_16  avg=84.0  total=420
  2. Student_12  avg=81.8  total=409
  3. Student_15  avg=78.2  total=391
  4. Student_17  avg=77.6  total=388
  5. Student_04  avg=76.0  total=380
Bottom 5:
  1. Student_25  avg=50.0  total=250
  2. Student_09  avg=56.4  total=282
  3. Student_27  avg=57.6  total=288
  4. Student_18  avg=59.0  total=295
  5. Student_23  avg=59.0  total=295


In [6]:
# Boolean mask: flag students failing ANY subject
fail_mask  = scores < 60                    # (30,5) boolean
any_fail   = fail_mask.any(axis=1)          # (30,) True where student fails at least one
multi_fail = fail_mask.sum(axis=1) >= 2     # True where student fails 2+ subjects

print(f"Students failing at least 1 subject: {any_fail.sum()}")
print(f"Students failing 2+ subjects:        {multi_fail.sum()}")

print("At-risk students (failing 2+ subjects):")
for idx in np.where(multi_fail)[0]:
    fail_subjects = np.array(["Mathematics","Python","Statistics","DS","Comm"])[fail_mask[idx]]
    print(f"  {student_names[idx]}  avg={scores[idx].mean():.1f}  failing: {list(fail_subjects)}")

Students failing at least 1 subject: 26
Students failing 2+ subjects:        21
At-risk students (failing 2+ subjects):
  Student_02  avg=70.8  failing: [np.str_('Mathematics'), np.str_('Python')]
  Student_03  avg=72.0  failing: [np.str_('Statistics'), np.str_('Comm')]
  Student_06  avg=60.8  failing: [np.str_('Mathematics'), np.str_('Statistics'), np.str_('DS')]
  Student_07  avg=70.8  failing: [np.str_('Statistics'), np.str_('DS')]
  Student_08  avg=61.0  failing: [np.str_('Mathematics'), np.str_('Statistics')]
  Student_09  avg=56.4  failing: [np.str_('Mathematics'), np.str_('Python'), np.str_('Statistics'), np.str_('DS')]
  Student_10  avg=72.2  failing: [np.str_('Mathematics'), np.str_('DS')]
  Student_13  avg=72.6  failing: [np.str_('Mathematics'), np.str_('Comm')]
  Student_14  avg=59.2  failing: [np.str_('DS'), np.str_('Comm')]
  Student_18  avg=59.0  failing: [np.str_('Mathematics'), np.str_('Python'), np.str_('Comm')]
  Student_19  avg=60.4  failing: [np.str_('Statistics'), 

In [7]:
# Normalise scores to [0, 1] per subject using broadcasting
mins  = scores.min(axis=0)    # shape (5,) - one min per subject
maxes = scores.max(axis=0)    # shape (5,)
norm  = (scores - mins) / (maxes - mins)

# Correlation between subjects
corr = np.corrcoef(scores.T)   # transpose so rows = subjects
subjects = ["Math","Python","Stats","DS","Comm"]

print("=== Inter-subject Correlation ===")
print(f"{'':>8}" + "".join(f"{s:>8}" for s in subjects))
for i, row in enumerate(corr):
    print(f"{subjects[i]:>8}" + "".join(f"{v:>8.3f}" for v in row))

# Simulate 10,000 exam outcomes and estimate P(score > 80 in all 5 subjects)
np.random.seed(42)
sim_means = scores.mean(axis=0)
sim_stds  = scores.std(axis=0)
simulated = np.random.normal(sim_means, sim_stds, (10_000, 5)).clip(0, 100)
all_pass  = (simulated > 80).all(axis=1)
print(f"P(score > 80 in ALL 5 subjects): {all_pass.mean()*100:.1f}%")

=== Inter-subject Correlation ===
            Math  Python   Stats      DS    Comm
    Math   1.000   0.073  -0.060  -0.002  -0.166
  Python   0.073   1.000  -0.139   0.018  -0.103
   Stats  -0.060  -0.139   1.000   0.288   0.130
      DS  -0.002   0.018   0.288   1.000  -0.256
    Comm  -0.166  -0.103   0.130  -0.256   1.000
P(score > 80 in ALL 5 subjects): 0.1%


## Summary

| Concept | Key point |
|---|---|
| Boolean indexing (RHS) | `a[a > 5]` returns a COPY of matching elements |
| Boolean assignment (LHS) | `a[a > 5] = 0` modifies in-place - the mask is on the LEFT |
| `np.where(cond, t, f)` | Vectorized if/else - returns new array |
| `np.argsort()` | Returns INDICES that sort the array - use them to reorder other arrays |
| `np.unique(a, return_counts=True)` | Distinct values and their frequencies |
| `np.random.seed(n)` | Set before every simulation - reproducibility is scientific integrity |
| `np.percentile(a, q)` | Robust quantile-based statistics - unaffected by extreme values |
| `np.corrcoef(matrix)` | Pass rows as variables - result[i,j] is correlation between row i and row j |

**Module 2 complete.** NumPy is now your computation engine. Everything that follows - Pandas, Matplotlib, statistics - uses NumPy arrays internally. When something runs slow in Pandas, you'll drop down to NumPy. When a visualisation needs transformation, you'll use NumPy. It's the foundation that never goes away.

Revise using the Module 2 question bank before the assessment.